# B_model: tests for `src/model.py` (Member B)

None of these cells need the CIFAR-10 data. Run with the **Python (cifar)** kernel.

## Environment check

In [ ]:
import sys, torch, torchvision, numpy
print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| torchvision:", torchvision.__version__, "| GPU:", torch.cuda.is_available())

## Cell 1: imports

In [ ]:
import sys
sys.path.append("../..")                    # lets Python find the src/ package

import torch
import torch.nn as nn
from src.model import MLP, count_parameters, parameter_table

torch.manual_seed(42)
model = MLP()
print(model)

## Cell 2: exactly three trainable layers, and the right parameter count

In [ ]:
n_linear = sum(isinstance(m, nn.Linear) for m in model.modules())
print("Linear layers:", n_linear)                         # 3
assert n_linear == 3

for name, shape, count in parameter_table(model):
    print(f"{name:<12} {str(shape):<14} {count:>10,}")
print("Total:", f"{count_parameters(model):,}")          # 1,707,274
assert count_parameters(model) == 1_707_274

## Cell 3: input and output shapes

In [ ]:
x = torch.randn(8, 3, 32, 32)               # a fake batch of 8 images
out = model(x)
print(out.shape)                            # torch.Size([8, 10])
assert out.shape == (8, 10)
print("row 0 sums to", out[0].sum().item())   # NOT 1.0: these are logits, not probabilities

## Cell 4: the loss works and gradients flow to every layer
An initial loss near 2.3 means the untrained model guesses evenly across 10 classes.

In [ ]:
y = torch.randint(0, 10, (8,))
loss = nn.CrossEntropyLoss()(out, y)
print("initial loss: %.3f (expect about ln(10) = 2.303)" % loss.item())
loss.backward()
for name, p in model.named_parameters():
    assert p.grad is not None and p.grad.abs().sum() > 0, f"no gradient in {name}"
print("gradients reach all 6 tensors")

## Cell 5: the model can memorise a tiny batch
If it can't learn 64 examples, it can't learn 45,000. Expect loss towards 0 and accuracy 1.00.

In [ ]:
torch.manual_seed(42)
model = MLP()
x = torch.randn(64, 3, 32, 32)
y = torch.randint(0, 10, (64,))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for step in range(200):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

acc = (model(x).argmax(1) == y).float().mean().item()
print("final loss %.4f, accuracy on the 64 examples: %.2f" % (loss.item(), acc))

## Cell 6: parameter counts of the configs compared on Day 2

In [ ]:
for h1, h2 in [(512, 256), (1024, 512)]:
    print(h1, h2, f"{count_parameters(MLP(h1, h2)):,}")
# 512 256  1,707,274
# 1024 512 3,676,682